# nb05 - Structured Streaming: files in, Delta out

The `data-generator` service drops JSON files into `/data/bronze/events_stream` every
few seconds (200 events each), with two landmines baked in:

- **~3% late events** - `ts` shifted up to 30 minutes into the past (watermark practice)
- **~2% duplicates** - a repeated `event_id` (dedup practice)

The files are written to a temp name and atomically renamed, so the file source never
sees half a file - exactly how a landing zone behaves in production.

**You will learn:** directory streams with `maxFilesPerTrigger`, explicit stream schemas,
watermarks, `dropDuplicatesWithinWatermark`, windowed aggregation, `foreachBatch` +
Delta MERGE, checkpoints, and `Trigger.AvailableNow` - every query here is **bounded**
so the notebook always terminates (the production version would just run forever).

**Prerequisite:** the stack is up. nb02 is *not* required (events land in their own
namespace); the generator has been running since `docker compose up`.


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

from de_stack import get_spark, BRONZE, CHECKPOINTS, register_table

spark = get_spark(
    "nb05-streaming",
    shuffle_partitions=8,
    # opt in to the 4.2 query-evolution features (named sources are experimental):
    extra_conf={"spark.sql.streaming.queryEvolution.enableSourceEvolution": "true"},
)
print("generator files so far:", len(spark.read.json(f"{BRONZE}/events_stream").inputFiles()))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 11:39:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


generator files so far: 111


## 1. The stream

Explicit schema, always: inference would re-read a sample of files on every restart,
and micro-batches would each have to guess. Note the nested structs and the
optional `order_id`/`amount` (only present on commerce events).


In [2]:
schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("ts", DoubleType()),                    # epoch seconds
    StructField("session_id", StringType()),
    StructField("user_id", StringType()),
    StructField("device", StringType()),
    StructField("page", StringType()),
    StructField("geo", StructType([
        StructField("country", StringType()),
        StructField("city", StringType()),
    ])),
    StructField("properties", StructType([
        StructField("price", DoubleType()),
        StructField("items", IntegerType()),
        StructField("ab_test_group", StringType()),
    ])),
    StructField("order_id", StringType()),              # absent on most events
    StructField("amount", DoubleType()),
])

raw_events = (
    spark.readStream
    .name("events_src")                                # 4.2: stable source name (experimental)
    .option("maxFilesPerTrigger", 200)                 # bound each micro-batch
    .schema(schema)
    .json(f"{BRONZE}/events_stream")
)

print("is streaming:", raw_events.isStreaming)


is streaming: True


## 2. Explore without writing: memory sink + `Trigger.AvailableNow`

`availableNow` processes everything currently available and **stops** - the perfect
bounded "run the stream once and look at it" mode (it is also how you would backfill).
The memory sink makes the result a plain temp view.


In [3]:
parsed = (
    raw_events
    .withColumn("event_ts", F.to_timestamp("ts"))
    .withColumn("country", F.col("geo.country"))
    .drop("ts", "geo")
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])        # 4.2-safe dedup (state + watermark)
)

probe = (parsed.writeStream.format("memory")
         .queryName("events_probe")
         .outputMode("append")
         .trigger(availableNow=True)
         .start())
probe.awaitTermination()

print("last batch rows:", probe.lastProgress["numInputRows"])
print("total rows     :", spark.sql("SELECT COUNT(*) FROM events_probe").first()[0])
spark.sql('''
    SELECT event_type, COUNT(*) AS n, COUNT(DISTINCT event_id) AS ids,
           MIN(event_ts) AS first_seen, MAX(event_ts) AS last_seen
    FROM events_probe GROUP BY event_type ORDER BY n DESC
''').show(truncate=False)


26/10/13 11:39:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e8a75357-19dc-44e7-a8f6-ab0d0a190dac. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/10/13 11:39:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/10/13 11:39:24 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


last batch rows: 0


26/10/13 11:39:33 WARN TaskSetManager: Stage 7 contains a task of very large size (1925 KiB). The maximum recommended task size is 1000 KiB.


total rows     : 22800


26/10/13 11:39:34 WARN TaskSetManager: Stage 10 contains a task of very large size (1925 KiB). The maximum recommended task size is 1000 KiB.


+-----------------+-----+-----+--------------------------+--------------------------+
|event_type       |n    |ids  |first_seen                |last_seen                 |
+-----------------+-----+-----+--------------------------+--------------------------+
|page_view        |10343|10343|2026-10-13 11:04:02.579178|2026-10-13 11:39:23.555126|
|add_to_cart      |5635 |5635 |2026-10-13 11:04:47.954885|2026-10-13 11:39:23.555126|
|order_created    |3365 |3365 |2026-10-13 11:04:53.025466|2026-10-13 11:39:23.555126|
|payment_processed|2295 |2295 |2026-10-13 11:04:33.724609|2026-10-13 11:39:23.555126|
|refund_issued    |1162 |1162 |2026-10-13 11:06:37.341679|2026-10-13 11:39:23.555126|
+-----------------+-----+-----+--------------------------+--------------------------+



Late events (backdated `ts`) inside the watermark window survive; state shows what the
dedup operator is holding. Duplicates arriving *later than* the watermark cannot be
caught by state - that is the fundamental trade-off you just configured.


In [4]:
state = probe.lastProgress.get("stateOperators", [{}])[0]
print("rows dropped by watermark:", state.get("numRowsDroppedByWatermark", 0))
print("state rows retained      :", state.get("numRowsTotal", 0))
print("memory rows (probe)      :", spark.sql("SELECT COUNT(*) FROM events_probe").first()[0])


rows dropped by watermark: 0
state rows retained      : 22522


memory rows (probe)      : 22800


26/10/13 11:39:36 WARN TaskSetManager: Stage 16 contains a task of very large size (1925 KiB). The maximum recommended task size is 1000 KiB.


## 3. Production shape: `foreachBatch` + Delta MERGE

The `memory` sink is for exploring. Production wants an **idempotent** sink: each
micro-batch MERGEs into a Delta table on `event_id`, so a replayed batch (crash between
commit and offset update) cannot create duplicates. Checkpoints remember which files
were already consumed.

> **New in 4.2:** the reader above carries a stable source name
> (`.name("events_src")`, experimental - names are letters/digits/underscores only).
> Named sources (and SQL's `IDENTIFIED BY` for sinks) let you add, remove or reorder stream components *without invalidating the
> checkpoint* - the offsets stay keyed to the name instead of the plan's position.
> Keep the name stable once a checkpoint exists.


In [5]:
silver_events = parsed.select(
    "event_id", "event_type", "event_ts", "session_id", "user_id", "device",
    "page", "country", "properties", "order_id", "amount",
)

# create the empty Delta table once
if not register_table(spark, "silver", "events"):
    spark.sql("CREATE DATABASE IF NOT EXISTS silver")
    (spark.createDataFrame([], silver_events.schema).write.format("delta")
        .saveAsTable("silver.events"))
    print("silver.events created (empty)")

def upsert_events(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    (DeltaTable.forName(spark, "silver.events").alias("t")
     .merge(batch_df.alias("s"), "t.event_id = s.event_id")
     .whenNotMatchedInsertAll()
     .execute())
    print(f"batch {batch_id} merged")


silver.events created (empty)


In [6]:
stream_q = (silver_events.writeStream
            .queryName("events-to-silver")                     # name shows in the Streaming tab
            .foreachBatch(upsert_events)
            .option("checkpointLocation", f"{CHECKPOINTS}/events_silver")
            .trigger(availableNow=True)
            .start())
stream_q.awaitTermination()

print("query:", stream_q.name, "|", stream_q.status["message"])


26/10/13 11:39:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/10/13 11:39:41 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


26/10/13 11:39:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


batch 0 merged


query: events-to-silver | Stopped


## 4. Idempotency, proven

Re-run the same query with the **same checkpoint**: it resumes, finds no new files,
touches nothing. Then replay everything with a **fresh checkpoint**: every file is
re-processed, but the MERGE on `event_id` means the table row count still only grows
by the genuinely new events.


In [7]:
rows_before = spark.sql("SELECT COUNT(*) FROM silver.events").first()[0]

stream_q2 = (silver_events.writeStream
             .queryName("events-to-silver")
             .foreachBatch(upsert_events)
             .option("checkpointLocation", f"{CHECKPOINTS}/events_silver")   # same checkpoint
             .trigger(availableNow=True)
             .start())
stream_q2.awaitTermination()

rows_after = spark.sql("SELECT COUNT(*) FROM silver.events").first()[0]
print(f"resume with same checkpoint: {rows_before} -> {rows_after} rows (new files only)")


26/10/13 11:40:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/10/13 11:40:04 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


batch 2 merged


resume with same checkpoint: 23800 -> 25374 rows (new files only)


In [8]:
# full replay from scratch: new checkpoint, all files re-read, MERGE dedups
stream_q3 = (silver_events.writeStream
             .queryName("events-to-silver-replay")
             .foreachBatch(upsert_events)
             .option("checkpointLocation", f"{CHECKPOINTS}/events_silver_replay")
             .trigger(availableNow=True)
             .start())
stream_q3.awaitTermination()

rows_replay = spark.sql("SELECT COUNT(*) FROM silver.events").first()[0]
print(f"full replay: {rows_after} -> {rows_replay} rows (only genuinely new events added)")


26/10/13 11:40:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/10/13 11:40:16 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


batch 0 merged


full replay: 25374 -> 26200 rows (only genuinely new events added)


## 5. What production changes

Swap `trigger(availableNow=True)` for `trigger(processingTime="30 seconds")` (or
4.1+'s experimental `trigger(realTime=...)` for sub-second stateless queries) and the
query runs forever; swap the file directory for a Kafka topic and the same operators
apply. State and progress live in the Streaming tab of the driver UI
(http://localhost:4041) - open it while the queries above ran.

Next: **nb06_spark4_feature_lab.ipynb** - VARIANT, geospatial types, Arrow UDFs and
SQL CDC on top of the layers we just built.
